# Tone Classifier

Fine-tunes `microsoft/deberta-v3-base` on labeled tone-rejected captions.

**Inputs:**
- `accepted_rejected.csv`

**Outputs saved to `/kaggle/working`:**
- `tone_model_dir/` — merged model + tokenizer via `save_pretrained()`
- `tone_artefacts.pkl` — threshold, config

**Settings (small-dataset optimised — ~105 tone-rejected captions):**
- `deberta-v3-base` (86M params), 3-fold CV
- Bottom 8/12 layers frozen, dropout 0.2, weight decay 0.05
- F2 threshold selection (recall-biased)


## 1. Installs

In [1]:
!pip install transformers torch scikit-learn tqdm --quiet


## 2. Imports & device

In [2]:
import gc
import math
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_fscore_support, accuracy_score,
    confusion_matrix, classification_report,
)
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR = '/kaggle/working'
print('Using device:', device)


Using device: cuda


## 3. Configuration

In [3]:
TONE_CONFIG = {
    'model':         'microsoft/deberta-v3-base',
    'lr':            5e-6,
    'epochs':        8,
    'batch':         8,
    'max_len':       128,
    'dropout':       0.2,
    'weight_decay':  0.05,
    'freeze_layers': 8,
}

LABEL_SMOOTHING = 0.1
POS_WEIGHT_CAP  = 10.0
THRESHOLD_SPLIT = 0.15
THRESHOLD_BETA  = 2.0   # F2: recall-biased for tone
N_FOLDS         = 3
SEED            = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


## 4. Load & clean dataset

In [4]:
import re
import emoji
import unicodedata

def remove_unicode_font_chars(text):
    # Mathematical Bold capitals A-Z: U+1D400–U+1D419
    # Mathematical Bold smalls a-z:   U+1D41A–U+1D433
    # Mathematical Bold Italic, Script, Fraktur, etc. follow in sequence
    # Fastest approach: re-encode to ASCII, replacing anything unresolvable
    result = []
    for char in text:
        # If NFKC decomposition gives us a plain ASCII letter, use it
        normalized = unicodedata.normalize("NFKC", char)
        if normalized.isascii():
            result.append(normalized)
        else:
            # Try to get the unicode name and extract the base letter
            # e.g. "MATHEMATICAL BOLD CAPITAL A" -> "A"
            try:
                name = unicodedata.name(char, "")
                if "MATHEMATICAL" in name or "BOLD" in name or "ITALIC" in name:
                    # Last word in the name is usually the base letter/digit
                    base = name.split()[-1]
                    if len(base) == 1:
                        result.append(base)
                    # else skip it
                else:
                    result.append(char)
            except Exception:
                result.append(char)
    return "".join(result)


def clean_text(text):
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2013", "-").replace("\u2014", "-")
    text = text.replace("\u2026", "...")
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = emoji.replace_emoji(text, replace='')
    text = text.replace('\ufffd', '')
    text = re.sub(r"[^a-zA-Z0-9\s.,!?'\-:;()/]", ' ', text)
    # Remove apostrophes not flanked by letters on both sides
    # keeps: it's, won't, Philippines'  |  removes: ' ' ' ' isolated artifacts
    text = re.sub(r"(?<![a-zA-Z])'|'(?![a-zA-Z])", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def clean_for_models(text):
    text = str(text)

    # 1. Remove full hashtag tokens FIRST, while # is still present
    text = re.sub(r'#\w+', '', text)

    # 2. Remove lead keywords
    text = re.sub(r'^(LOOK|READ|WATCH|CHECK THIS OUT|LISTEN|ICYMI|PSA)\s*[|:\-]?\s*',
                  '', text, flags=re.IGNORECASE)
    text = clean_text(text)
    return text


def clean_for_rules(text):
    """
    Minimal cleaning for formatting/structure rule-based features.
    Preserves |, #, capitalization patterns, lead keywords.
    """
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.replace("\u2013", "-").replace("\u2014", "-")
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = emoji.replace_emoji(text, replace='')
    text = text.replace('\ufffd', '')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [5]:
df = pd.read_csv('/kaggle/input/datasets/mariellemayol/accepted-rejected/accepted_rejected_v2.csv')
df['text']   = df['text'].astype(str).str.strip()
df['status'] = df['status'].str.lower().str.strip()
df['issue']  = df['issue'].fillna('NA').str.lower().str.strip()
df['text'] = df['text'].apply(clean_for_models)

# ── Deduplicate on normalised text BEFORE any split ──────────────────
_norm = df['text'].str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()
n_before = len(df)
df = df[~_norm.duplicated(keep='first')].reset_index(drop=True)
print(f'Deduplicated: {n_before} → {len(df)} rows ({n_before - len(df)} duplicates removed)')

# Remove entries shorter than 10 characters
before_len = len(df)
df = df[df['text'].str.len() >= 10].reset_index(drop=True)
removed_short = before_len - len(df)
print(f'Short text filter: removed {removed_short} entries with < 10 characters ({before_len} → {len(df)})')

print('Dataset shape:', df.shape)
print('\nStatus distribution:')
print(df['status'].value_counts())
print('\nIssue distribution (rejected only):')
print(df[df['status'] == 'rejected']['issue'].value_counts())
display(df.head())


Deduplicated: 1822 → 1804 rows (18 duplicates removed)
Short text filter: removed 2 entries with < 10 characters (1804 → 1802)
Dataset shape: (1802, 3)

Status distribution:
status
accepted    1396
rejected     406
Name: count, dtype: int64

Issue distribution (rejected only):
issue
grammar        197
tone           105
inclusivity    104
Name: count, dtype: int64


,text,status,issue
0,ADVANCING TRANSPARENT YOUTH GOVERNANCE IN THE ...,accepted,na
1,THE CITY OF SAN JOSE DEL MONTE STRENGTHENS YOU...,accepted,na
2,Y-GOV 2026 TRAINING TO STRENGTHEN THE CAPACITY...,accepted,na
3,WE ARE HIRING! The National Youth Commission i...,accepted,na
4,SK Advisory No. 2026-NP003 : The National Yout...,accepted,na


## 5. Build tone dataset

In [6]:
def build_tone_df(df):
    accepted       = df[df['status'] == 'accepted'].copy()
    rejected_tone  = df[(df['status'] == 'rejected') & (df['issue'] == 'tone')].copy()
    rejected_other = df[(df['status'] == 'rejected') & (df['issue'] != 'tone')].copy()
    accepted['label']       = 1
    rejected_other['label'] = 1
    rejected_tone['label']  = 0
    full = pd.concat([accepted, rejected_other, rejected_tone], ignore_index=True)
    cv_df, thresh_df = train_test_split(
        full, test_size=THRESHOLD_SPLIT, stratify=full['label'], random_state=SEED,
    )
    cv_df = cv_df.reset_index(drop=True)
    thresh_df = thresh_df.reset_index(drop=True)
    print(f'CV pool     : {len(cv_df)}  (pos={cv_df.label.sum()}, neg={(cv_df.label==0).sum()})')
    print(f'Thresh split: {len(thresh_df)}  (pos={thresh_df.label.sum()}, neg={(thresh_df.label==0).sum()})')
    print(f'Neg per fold (~{N_FOLDS}): ~{int((cv_df.label==0).sum()) // N_FOLDS}')
    return cv_df[['text','label']], thresh_df[['text','label']]

cv_df, thresh_df = build_tone_df(df)


CV pool     : 1531  (pos=1442, neg=89)
Thresh split: 271  (pos=255, neg=16)
Neg per fold (~3): ~29


## 6. Dataset class, model utilities, training helpers

In [7]:
class CaptionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.float)
        }

In [8]:
def patch_classifier_head(model, dropout=0.1):
    """
    Replace default classifier head with Linear → Dropout(dropout).
    Dropout value is configurable per dimension.
    """
    old = model.classifier
    in_features = old.in_features if hasattr(old, 'in_features') else old.weight.shape[1]
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 1),
        nn.Dropout(dropout),
    )
    model = model.float().to(device)
    nn.init.xavier_uniform_(model.classifier[0].weight)
    nn.init.zeros_(model.classifier[0].bias)
    return model


def freeze_bottom_layers(model, n_freeze):
    """
    Freeze the bottom n_freeze transformer layers and the embedding layer.
    Only applies to deberta-v3 architecture; safely skips for other models.
    """
    if n_freeze <= 0:
        return

    try:
        encoder_layers = model.deberta.encoder.layer
        n_layers       = len(encoder_layers)
        n_freeze       = min(n_freeze, n_layers)

        for i in range(n_freeze):
            for param in encoder_layers[i].parameters():
                param.requires_grad = False

        for param in model.deberta.embeddings.parameters():
            param.requires_grad = False

        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in model.parameters())
        print(f'  Frozen {n_freeze}/{n_layers} encoder layers + embeddings '
              f'| Trainable: {trainable:,}/{total:,} ({100*trainable/total:.1f}%)')
    except AttributeError:
        print('  Layer freezing skipped (model architecture not deberta-v3)')


In [9]:
def smooth_labels(labels, smoothing=LABEL_SMOOTHING):
    return labels * (1.0 - smoothing) + 0.5 * smoothing

def oversample_minority(tr_texts, tr_labels, min_count=100, factor=2):
    labels_arr     = np.array(tr_labels)
    n_pos          = int((labels_arr == 1).sum())
    n_neg          = int((labels_arr == 0).sum())
    minority_label = 0 if n_neg < n_pos else 1
    minority_count = min(n_pos, n_neg)
    if minority_count >= min_count:
        return list(tr_texts), list(tr_labels)
    minority_idx    = [i for i, l in enumerate(tr_labels) if l == minority_label]
    extra_idx       = minority_idx * (factor - 1)
    combined_texts  = list(tr_texts)  + [tr_texts[i] for i in extra_idx]
    combined_labels = list(tr_labels) + [tr_labels[i] for i in extra_idx]
    print(f'    Oversampled minority (label={minority_label}): {minority_count} → {minority_count*factor}')
    return combined_texts, combined_labels

In [10]:
def release_memory(*objects):
    for obj in objects:
        try:
            del obj
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        print(f'  GPU: {torch.cuda.memory_allocated()/1e9:.2f}GB alloc, '
              f'{torch.cuda.memory_reserved()/1e9:.2f}GB reserved')

In [11]:
def train_one_fold(model, train_loader, val_loader, optimizer, scheduler,
                   criterion, epochs, dimension, fold):
    best_auc   = 0.0
    best_state = None

    for epoch in range(epochs):
        model.train()
        running_loss    = 0.0
        skipped_batches = 0

        for batch in tqdm(train_loader,
                          desc=f'  [{dimension}] fold {fold+1} epoch {epoch+1}/{epochs}',
                          leave=False):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device).float()
            smoothed       = smooth_labels(labels)

            optimizer.zero_grad()

            assert next(model.parameters()).dtype == torch.float32

            logits = model(input_ids=input_ids,
                           attention_mask=attention_mask).logits.squeeze(-1)

            if torch.isnan(logits).any() or torch.isinf(logits).any():
                skipped_batches += 1
                continue

            loss = criterion(logits, smoothed)

            if torch.isnan(loss) or torch.isinf(loss):
                skipped_batches += 1
                continue

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            running_loss += loss.item()

        if skipped_batches > 0:
            print(f'  WARNING: {skipped_batches} batches skipped in fold {fold+1} epoch {epoch+1}')

        # Validate
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for batch in val_loader:
                logits = model(
                    input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device)
                ).logits.squeeze(-1)
                probs = torch.sigmoid(logits).cpu().numpy()
                probs = np.nan_to_num(probs, nan=0.5)
                val_preds.extend(probs)
                val_true.extend(batch['label'].numpy())

        val_true_arr  = np.array(val_true)
        val_preds_arr = np.array(val_preds)

        if len(np.unique(val_true_arr)) < 2:
            print(f'  WARNING: only one class in val fold {fold+1} — skipping AUC')
            continue

        auc      = roc_auc_score(val_true_arr, val_preds_arr)
        avg_loss = running_loss / max(len(train_loader) - skipped_batches, 1)
        print(f'  [{dimension}] fold {fold+1} | epoch {epoch+1}/{epochs} '
              f'| loss {avg_loss:.4f} | val AUC {auc:.4f}')

        if auc > best_auc:
            best_auc   = auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    del optimizer, scheduler
    gc.collect()
    return best_state, best_auc

## 7. Fine-tuning loop (3-fold)

In [12]:
texts  = cv_df['text'].tolist()
labels = cv_df['label'].values
n_pos  = labels.sum()
n_neg  = len(labels) - n_pos
pos_weight = torch.tensor(
    [min(n_neg / n_pos, POS_WEIGHT_CAP)], dtype=torch.float
).to(device)
print(f'Class balance — pos: {n_pos}, neg: {n_neg}  |  pos_weight: {pos_weight.item():.2f}x')

tokenizer = AutoTokenizer.from_pretrained(TONE_CONFIG['model'])
skf       = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_probs = np.zeros(len(cv_df))
fold_aucs = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(texts, labels)):
    print(f"\n{'='*55}\n  Fold {fold+1}/{N_FOLDS}\n{'='*55}")
    tr_texts, tr_labels   = [texts[i] for i in tr_idx], list(labels[tr_idx])
    val_texts, val_labels = [texts[i] for i in val_idx], labels[val_idx]
    tr_texts, tr_labels   = oversample_minority(tr_texts, tr_labels)

    train_ds = CaptionDataset(tr_texts,  tr_labels,  tokenizer, TONE_CONFIG['max_len'])
    val_ds   = CaptionDataset(val_texts, val_labels, tokenizer, TONE_CONFIG['max_len'])
    train_loader = DataLoader(train_ds, batch_size=TONE_CONFIG['batch'],   shuffle=True,  num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=TONE_CONFIG['batch']*2, shuffle=False, num_workers=0, pin_memory=True)

    model = AutoModelForSequenceClassification.from_pretrained(TONE_CONFIG['model'], num_labels=1).to(device)
    model = patch_classifier_head(model, dropout=TONE_CONFIG['dropout'])
    freeze_bottom_layers(model, TONE_CONFIG['freeze_layers'])

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=TONE_CONFIG['lr'], weight_decay=TONE_CONFIG['weight_decay'],
    )
    total_steps = len(train_loader) * TONE_CONFIG['epochs']
    scheduler   = get_cosine_schedule_with_warmup(optimizer,
                      num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

    best_state, best_auc = train_one_fold(
        model, train_loader, val_loader, optimizer, scheduler,
        criterion, TONE_CONFIG['epochs'], 'tone', fold,
    )
    torch.save(best_state, f'{SAVE_DIR}/tone_fold{fold+1}.pth')
    print(f'  Saved fold {fold+1}  (AUC {best_auc:.4f})')

    # OOF
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            logits = model(input_ids=batch['input_ids'].to(device),
                           attention_mask=batch['attention_mask'].to(device)).logits.squeeze(-1)
            probs  = np.nan_to_num(torch.sigmoid(logits).cpu().numpy(), nan=0.5)
            start  = i * val_loader.batch_size
            oof_probs[val_idx[start:start+len(probs)]] = probs

    fold_aucs.append(best_auc)
    release_memory(model, train_loader, val_loader, train_ds, val_ds, best_state, criterion)

print(f'\nFold AUCs : {[round(a,4) for a in fold_aucs]}')
print(f'Mean AUC  : {np.mean(fold_aucs):.4f}  |  Std: {np.std(fold_aucs):.4f}')
release_memory(tokenizer, pos_weight)


Class balance — pos: 1442, neg: 89  |  pos_weight: 0.06x


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]


  Fold 1/3
    Oversampled minority (label=0): 59 → 118


...

  [tone] fold 3 | epoch 4/8 | loss 0.0926 | val AUC 0.9921


  [tone] fold 3 | epoch 5/8 | loss 0.0896 | val AUC 0.9947


  [tone] fold 3 | epoch 6/8 | loss 0.0898 | val AUC 0.9811


  [tone] fold 3 | epoch 7/8 | loss 0.0864 | val AUC 0.9795


  [tone] fold 3 | epoch 8/8 | loss 0.0883 | val AUC 0.9733
  Saved fold 3  (AUC 0.9947)
  GPU: 1.15GB alloc, 1.70GB reserved

Fold AUCs : [np.float64(0.9735), np.float64(0.9847), np.float64(0.9947)]
Mean AUC  : 0.9843  |  Std: 0.0087
  GPU: 1.15GB alloc, 1.70GB reserved


## 8. Threshold selection & OOF evaluation

In [13]:
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report

def scale_score(raw_prob, threshold):
    import math
    def _sig(x, lo, hi):
        s = 1 / (1 + math.exp(-10 * (x - 0.5)))
        return lo + (hi - lo) * s
    if raw_prob >= threshold:
        return round(_sig((raw_prob - threshold) / (1.0 - threshold + 1e-8), 70.0, 100.0), 1)
    return round(_sig(raw_prob / (threshold + 1e-8), 0.0, 50.0), 1)

def get_thresh_probs(thresh_df):
    tokenizer = AutoTokenizer.from_pretrained(TONE_CONFIG['model'])
    ds        = CaptionDataset(thresh_df['text'].tolist(), thresh_df['label'].tolist(),
                               tokenizer, TONE_CONFIG['max_len'])
    loader    = DataLoader(ds, batch_size=TONE_CONFIG['batch']*2, shuffle=False, num_workers=0)
    all_probs = []
    for fold in range(N_FOLDS):
        model = AutoModelForSequenceClassification.from_pretrained(TONE_CONFIG['model'], num_labels=1).to(device)
        model = patch_classifier_head(model, dropout=TONE_CONFIG['dropout'])
        state = torch.load(f'{SAVE_DIR}/tone_fold{fold+1}.pth', map_location=device)
        model.load_state_dict(state); model.eval()
        fold_probs = []
        with torch.no_grad():
            for batch in loader:
                logits = model(input_ids=batch['input_ids'].to(device),
                               attention_mask=batch['attention_mask'].to(device)).logits.squeeze(-1)
                fold_probs.extend(np.nan_to_num(torch.sigmoid(logits).cpu().numpy(), nan=0.5))
        all_probs.append(fold_probs)
        release_memory(model, state)
    release_memory(tokenizer)
    return np.mean(all_probs, axis=0)

def select_threshold(thresh_df, thresh_probs, beta=THRESHOLD_BETA):
    from sklearn.metrics import precision_recall_fscore_support
    y_true = thresh_df['label'].values
    best_t, best_score = 0.5, 0.0
    for t in np.arange(0.05, 0.96, 0.01):
        preds = (thresh_probs >= t).astype(int)
        prec, rec, _, _ = precision_recall_fscore_support(y_true, preds, average='binary', zero_division=0)
        denom = (beta**2 * prec + rec)
        fb    = (1 + beta**2) * prec * rec / denom if denom > 0 else 0.0
        if fb > best_score:
            best_score = fb; best_t = t
    return round(float(best_t), 2)

thresh_probs = get_thresh_probs(thresh_df)
threshold    = select_threshold(thresh_df, thresh_probs)
print(f'Threshold: {threshold}  (pass >= {threshold})')

# OOF eval
y_true = cv_df['label'].values
auc_roc = roc_auc_score(y_true, oof_probs)
auc_pr  = average_precision_score(y_true, oof_probs)
preds   = (oof_probs >= threshold).astype(int)
from sklearn.metrics import accuracy_score
acc     = accuracy_score(y_true, preds)
from sklearn.metrics import precision_recall_fscore_support
prec, rec, f1, _ = precision_recall_fscore_support(y_true, preds, average='binary', zero_division=0)
cm = confusion_matrix(y_true, preds)
print(f"\n{'='*50}\n  TONE OOF Evaluation\n{'='*50}")
print(f'  AUC-ROC: {auc_roc:.4f}  |  AUC-PR: {auc_pr:.4f}')
print(f'  Threshold: {threshold}  |  Accuracy: {acc:.4f}')
print(f'  Precision: {prec:.4f}  |  Recall: {rec:.4f}  |  F1: {f1:.4f}')
print(f'  Confusion matrix:\n    [[TN={cm[0,0]} FP={cm[0,1]}]\n     [FN={cm[1,0]} TP={cm[1,1]}]]')
print(classification_report(y_true, preds, target_names=['fail','pass'], zero_division=0))


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias          

  GPU: 2.63GB alloc, 2.71GB reserved


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias          

  GPU: 2.63GB alloc, 2.85GB reserved


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias          

  GPU: 2.63GB alloc, 3.05GB reserved
  GPU: 2.63GB alloc, 3.05GB reserved
Threshold: 0.08  (pass >= 0.08)

  TONE OOF Evaluation
  AUC-ROC: 0.9831  |  AUC-PR: 0.9988
  Threshold: 0.08  |  Accuracy: 0.9811
  Precision: 0.9856  |  Recall: 0.9945  |  F1: 0.9900
  Confusion matrix:
    [[TN=68 FP=21]
     [FN=8 TP=1434]]
              precision    recall  f1-score   support

        fail       0.89      0.76      0.82        89
        pass       0.99      0.99      0.99      1442

    accuracy                           0.98      1531
   macro avg       0.94      0.88      0.91      1531
weighted avg       0.98      0.98      0.98      1531



## 9. Model soup + save_pretrained

In [14]:
import os, pickle, shutil

def merge_folds(n_folds=N_FOLDS):
    running_mean, float_keys = None, None
    for fold in range(n_folds):
        path  = f'{SAVE_DIR}/tone_fold{fold+1}.pth'
        state = torch.load(path, map_location='cpu')
        if running_mean is None:
            running_mean = {k: v.clone().float() for k, v in state.items()}
            float_keys   = {k for k, v in state.items() if v.dtype.is_floating_point}
        else:
            n = fold + 1
            for key in float_keys:
                running_mean[key] += (state[key].float() - running_mean[key]) / n
        del state; gc.collect()
        os.remove(path)
        print(f'  Fold {fold+1} merged and deleted.')
    return running_mean

# Save best fold .pth BEFORE merge_folds deletes all fold files
best_fold_idx  = int(np.argmax(fold_aucs))          # 0-based index of best fold
best_fold_path = f'{SAVE_DIR}/tone_fold{best_fold_idx+1}.pth'
best_fold_save = f'{SAVE_DIR}/tone_best_fold.pth'
shutil.copy(best_fold_path, best_fold_save)
print(f'Best fold: {best_fold_idx+1}  (AUC {fold_aucs[best_fold_idx]:.4f})  → saved as tone_best_fold.pth')

print('Merging folds...')
merged_state = merge_folds()

# Rebuild model, load merged weights, save_pretrained
# (soup model kept for reference; inference will use best fold weights instead)
deploy_model = AutoModelForSequenceClassification.from_pretrained(TONE_CONFIG['model'], num_labels=1)
deploy_model = patch_classifier_head(deploy_model, dropout=TONE_CONFIG['dropout'])
deploy_model = deploy_model.float()
deploy_model.load_state_dict(merged_state)
deploy_model.eval()

model_dir = f'{SAVE_DIR}/tone_model_dir'
os.makedirs(model_dir, exist_ok=True)
deploy_model.save_pretrained(model_dir)
AutoTokenizer.from_pretrained(TONE_CONFIG['model']).save_pretrained(model_dir)
# Also copy best-fold weights into model_dir so Notebook 3 / app.py can find them
shutil.copy(best_fold_save, f'{model_dir}/best_fold.pth')
print(f'Saved tone model → {model_dir}')
release_memory(deploy_model, merged_state)

# Save artefacts for Notebook 3
artefacts = {
    'threshold':      threshold,
    'config':         TONE_CONFIG,
    'model_dir':      model_dir,
    'fold_aucs':      fold_aucs,
    'best_fold_idx':  best_fold_idx,
    'best_fold_path': f'{model_dir}/best_fold.pth',
    'scale_score':    scale_score,
}
with open(f'{SAVE_DIR}/tone_artefacts.pkl', 'wb') as f:
    pickle.dump(artefacts, f)
print(f'Saved artefacts → {SAVE_DIR}/tone_artefacts.pkl')

print('\nFiles in SAVE_DIR:')
for fn in sorted(os.listdir(SAVE_DIR)):
    print(f'  {fn:50s}  {os.path.getsize(f"{SAVE_DIR}/{fn}")/1e6:6.1f} MB')

Best fold: 3  (AUC 0.9947)  → saved as tone_best_fold.pth
Merging folds...
  Fold 1 merged and deleted.
  Fold 2 merged and deleted.
  Fold 3 merged and deleted.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias          

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved tone model → /kaggle/working/tone_model_dir
  GPU: 1.89GB alloc, 2.26GB reserved
Saved artefacts → /kaggle/working/tone_artefacts.pkl

Files in SAVE_DIR:
  __notebook__.ipynb                                     0.7 MB
  tone_artefacts.pkl                                     0.0 MB
  tone_best_fold.pth                                   737.8 MB
  tone_model_dir                                         0.0 MB
